In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, window, mean
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
df = spark.read.json("transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
from pyspark.sql.functions import col, window, avg, round as _round, desc, count

gdansk = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia")
    )
    .orderBy("srednia", ascending=True)
)

gdansk.show(1)

+--------------------+-------+
|              window|srednia|
+--------------------+-------+
|{2026-04-12 08:00...| 395.01|
+--------------------+-------+
only showing top 1 row



In [13]:
kategorie = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") & 
        (col("timestamp") < "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id"))
    .orderBy("category")
)

kategorie.show()

+-----------+------------+
|   category|count(tx_id)|
+-----------+------------+
|elektronika|         611|
|    książki|         622|
|     odzież|         605|
|    żywność|         567|
+-----------+------------+



In [18]:
okno15 = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id"))
    .orderBy(desc(count("tx_id")))
)
okno15.show(1)

+--------------------+------------+
|              window|count(tx_id)|
+--------------------+------------+
|{2026-04-12 09:15...|        1234|
+--------------------+------------+
only showing top 1 row

